# Keithley 2460 narrow M5 buffer proof

This guarded notebook reproduces the target-buffer proof on a nominal 100 Ω resistor. It is **not** for a battery or unknown DUT. Use the front force and sense terminals in four-wire configuration, select TSP, and physically confirm that the output indicator is off. Every mutating phase is disabled by default.

The proof is limited to +1 mA and a 0.2 V limit. The runtime allocates a standard fill-once buffer for the requested number of 1 NPLC readings (subject to the instrument's 16-record minimum), with a 5,000,000-record instrument-side backstop; this notebook applies a much smaller explicit host ceiling. Acquisition may be immediate or wait locally for the commissioned external START input. Retrieval is non-destructive and chunked, archive output never overwrites, and discard is a separate explicit step. Buffer timestamps are relative to the first reading, not to actual START, so this notebook does not construct the shared M5 `MeasurementRecord` type or promote a hardware gate.

In [ ]:
from __future__ import annotations

import json
import statistics
import time
from dataclasses import asdict
from datetime import UTC, datetime

from ophyd_electrochemistry.acquisition import StartMode
from ophyd_electrochemistry.keithley.k2460 import (
    BufferPolicy,
    CurrentHoldAcquisitionProof,
    DigitalIOConfig,
    Keithley2460Config,
    M4RuntimeController,
    PyVisaKeithley2460Transport,
    RuntimeState,
    SafetyConfig,
    TimingPolicy,
    VisaTransportConfig,
    assemble_runtime_buffer_archive,
    packaged_runtime,
    runtime_buffer_archive_from_json,
    runtime_buffer_archive_json,
    write_runtime_buffer_archive,
)
from ophyd_electrochemistry.keithley.k2460.commissioning import run_read_only_diagnostic

RESOURCE = "TCPIP0::169.254.113.151::5025::SOCKET"
RUN_QUERY_ONLY_PREFLIGHT = False
INSTALL_OR_REPLACE_VOLATILE_RUNTIME = False
RUN_BUFFER_PROOF = False
RUN_EXTERNAL_START_BUFFER_PROOF = False
RUN_INDEPENDENT_AUDIT = False
DISCARD_RETAINED_BUFFER = False
CONFIRM_FRONT_4WIRE_100_OHM_AND_OUTPUT_OFF = False
MEASUREMENT_COUNT = 3  # Keep commissioning runs small; 17 exercises the former boundary.
BUFFER_CAPACITY_CEILING = 1_000
CHUNK_SIZE = 128
ACQUISITION_ID = "replace-with-a-unique-run-id"
ARCHIVE_DESTINATION = None  # Set a new path; existing files are never overwritten.

In [ ]:
artifact = packaged_runtime()
assert artifact.build == "m4-finite-current-hold-v12"
assert artifact.script_name == f"oe_m4_{artifact.sha256[:24]}"
assert "trigger.BLOCK_MEASURE_DIGITIZE" in artifact.source
print({"abi": artifact.abi, "build": artifact.build, "sha256": artifact.sha256})

In [ ]:
if not RUN_QUERY_ONLY_PREFLIGHT:
    raise RuntimeError("STOP: query-only hardware preflight is disabled by default.")

visa_config = VisaTransportConfig(
    resource_name=RESOURCE, backend="@py", required_language="TSP", io_timeout_ms=5_000
)
preflight = run_read_only_diagnostic(visa_config)
assert preflight.output_enabled is False
print(json.dumps(preflight.as_dict(), indent=2, sort_keys=True))

In [ ]:
config = Keithley2460Config(
    visa_resource=RESOURCE,
    source_terminal="front",
    sense="remote",
    io=DigitalIOConfig(start_edge="either"),
    buffer=BufferPolicy(
        capacity_ceiling_records=BUFFER_CAPACITY_CEILING,
        transfer_chunk_records=CHUNK_SIZE,
    ),
    safety=SafetyConfig(
        voltage_min_v=-0.2,
        voltage_max_v=0.2,
        charge_current_max_a=0.001,
        discharge_current_max_a=0.001,
        power_abs_max_w=0.001,
        source_off_mode="normal",
    ),
    timing=TimingPolicy(
        command_timeout_s=10,
        external_start_timeout_s=60,
        shutdown_timeout_s=2,
        poll_period_s=0.01,
        required_abort_latency_s=1,
        required_cutoff_latency_s=1,
    ),
)


def new_transport(io_timeout_ms: int = 10_000) -> PyVisaKeithley2460Transport:
    return PyVisaKeithley2460Transport(
        VisaTransportConfig(
            resource_name=RESOURCE,
            backend="@py",
            required_language="TSP",
            io_timeout_ms=io_timeout_ms,
            script_line_delay_s=0.01,
        )
    )

In [ ]:
if not CONFIRM_FRONT_4WIRE_100_OHM_AND_OUTPUT_OFF:
    raise RuntimeError("STOP: the physical resistor setup and output-off state are not confirmed.")
if not INSTALL_OR_REPLACE_VOLATILE_RUNTIME:
    raise RuntimeError("STOP: volatile runtime installation is disabled by default.")

transport = new_transport()
try:
    transport.connect()
    controller = M4RuntimeController(transport, config)
    loaded_build = transport.query("print(oe_m4_runtime_build)")
    if loaded_build == artifact.build:
        assert transport.query("print(oe_m4_runtime_abi)") == artifact.abi
        assert transport.query(f"print({artifact.script_name} == nil)") == "false"
        installed = artifact
    else:
        installed = controller.replace_known_runtime_and_install()
    status = controller.status()
    assert status.state == RuntimeState.IDLE and not status.output_enabled
    assert float(transport.query("print(smu.source.level)")) == 0.0
    assert controller.buffer_info().record_count == 0
    print(
        json.dumps(
            {"runtime": asdict(installed), "status": asdict(status)},
            default=str,
            indent=2,
        )
    )
finally:
    transport.close()

In [ ]:
if not CONFIRM_FRONT_4WIRE_100_OHM_AND_OUTPUT_OFF:
    raise RuntimeError("STOP: the physical resistor setup and output-off state are not confirmed.")
if RUN_BUFFER_PROOF == RUN_EXTERNAL_START_BUFFER_PROOF:
    raise RuntimeError("STOP: enable exactly one immediate or external buffer proof.")

transport = new_transport()
controller = None
try:
    transport.connect()
    controller = M4RuntimeController(transport, config)
    installed = packaged_runtime()
    assert transport.query("print(oe_m4_runtime_build)") == installed.build
    assert transport.query(f"print({installed.script_name} == nil)") == "false"
    initial = controller.status()
    assert initial.state == RuntimeState.IDLE and not initial.output_enabled
    counts_before = {
        "warnings": int(transport.query("print(eventlog.getcount(eventlog.SEV_WARN))")),
        "errors": int(transport.query("print(eventlog.getcount(eventlog.SEV_ERROR))")),
    }
    request = CurrentHoldAcquisitionProof(
        current_a=0.001,
        source_range_a=0.001,
        voltage_limit_v=0.2,
        measurement_count=MEASUREMENT_COUNT,
        start_mode=(
            StartMode.EXTERNAL_TRIGGER if RUN_EXTERNAL_START_BUFFER_PROOF else StartMode.IMMEDIATE
        ),
    )
    prepared = controller.prepare_current_hold_acquisition(request)
    terminal = controller.arm()
    if request.start_mode == StartMode.EXTERNAL_TRIGGER:
        assert terminal.state == RuntimeState.WAITING_START and not terminal.output_enabled
        print("READY: make one START transition now", flush=True)
    deadline = time.monotonic() + 65.0
    while (
        terminal.state in (RuntimeState.RUNNING, RuntimeState.WAITING_START)
        and time.monotonic() < deadline
    ):
        time.sleep(0.02)
        terminal = controller.status()
    assert terminal.state == RuntimeState.COMPLETE and not terminal.output_enabled
    info = controller.buffer_info()
    chunks = []
    offset = 0
    while offset < info.record_count:
        chunk = controller.read_buffer_chunk(offset=offset, max_records=CHUNK_SIZE)
        chunks.append(chunk)
        offset = chunk.next_offset
    retry = controller.read_buffer_chunk(offset=chunks[0].offset, max_records=CHUNK_SIZE)
    assert chunks[0] == retry and chunks[-1].final
    recorded_at_utc = datetime.now(UTC).isoformat().replace("+00:00", "Z")
    archive = assemble_runtime_buffer_archive(
        acquisition_id=ACQUISITION_ID,
        recorded_at_utc=recorded_at_utc,
        instrument_identity=transport.identity.raw,
        runtime_build=installed.build,
        runtime_sha256=installed.sha256,
        runtime_script_name=installed.script_name,
        request=request,
        capacity_records=info.capacity_records,
        chunks=tuple(chunks),
    )
    archive_json = runtime_buffer_archive_json(archive)
    assert runtime_buffer_archive_from_json(archive_json) == archive
    archive_sha256 = json.loads(archive_json)["sha256"]
    if ARCHIVE_DESTINATION is not None:
        assert write_runtime_buffer_archive(ARCHIVE_DESTINATION, archive) == archive_sha256
    assert [record.buffer_relative_time_s for record in archive.records] == sorted(
        record.buffer_relative_time_s for record in archive.records
    )
    assert archive.records[0].buffer_relative_time_s == 0.0
    assert all(record.source_status & 8 for record in archive.records)  # source readback
    assert all(record.source_status & 64 for record in archive.records)  # four-wire sense
    assert all(record.source_status & 128 for record in archive.records)  # output on
    assert all(record.measurement_status & 8 for record in archive.records)  # front
    resistance_ohm = [
        record.measured_voltage_v / record.source_current_a for record in archive.records
    ]
    assert all(80.0 <= value <= 120.0 for value in resistance_ohm)
    recovered = controller.recover()  # retained data are intentionally preserved
    final_output = transport.query("print(smu.source.output)")
    final_level = float(transport.query("print(smu.source.level)"))
    counts_after = {
        "warnings": int(transport.query("print(eventlog.getcount(eventlog.SEV_WARN))")),
        "errors": int(transport.query("print(eventlog.getcount(eventlog.SEV_ERROR))")),
    }
    assert recovered.state == RuntimeState.IDLE and not recovered.output_enabled
    assert final_output == "smu.OFF" and final_level == 0.0
    assert counts_after == counts_before
    evidence = {
        "runtime": {
            "abi": installed.abi,
            "build": installed.build,
            "sha256": installed.sha256,
            "script_name": installed.script_name,
        },
        "request": asdict(request),
        "terminal": asdict(terminal),
        "buffer": asdict(info),
        "chunk_sizes": [len(chunk.records) for chunk in chunks],
        "archive_sha256": archive_sha256,
        "records_sha256": archive.records_sha256,
        "resistance_ohm": resistance_ohm,
        "mean_resistance_ohm": statistics.mean(resistance_ohm),
        "recovered": asdict(recovered),
        "event_counts_before": counts_before,
        "event_counts_after": counts_after,
    }
    print(json.dumps(evidence, default=str, indent=2, sort_keys=True))
except BaseException:
    if controller is not None and transport.connected:
        try:
            controller.force_safe()
        except BaseException:
            pass
    raise
finally:
    transport.close()

In [ ]:
if not RUN_INDEPENDENT_AUDIT:
    raise RuntimeError("STOP: independent audit is disabled by default.")

transport = new_transport(io_timeout_ms=5_000)
try:
    transport.connect()
    controller = M4RuntimeController(transport, config)
    status = controller.status()
    audit = {
        "status": asdict(status),
        "output": transport.query("print(smu.source.output)"),
        "source_level_a": float(transport.query("print(smu.source.level)")),
        "buffer": asdict(controller.buffer_info()),
    }
    assert status.state == RuntimeState.IDLE and not status.output_enabled
    assert audit["output"] == "smu.OFF" and audit["source_level_a"] == 0.0
    print(json.dumps(audit, default=str, indent=2, sort_keys=True))
finally:
    transport.close()

Only run the next cell after the complete record has been archived. Discard is destructive and cannot be undone.

In [ ]:
if not DISCARD_RETAINED_BUFFER:
    raise RuntimeError("STOP: destructive retained-buffer discard is disabled by default.")

transport = new_transport(io_timeout_ms=5_000)
try:
    transport.connect()
    controller = M4RuntimeController(transport, config)
    discarded = controller.discard_buffer(reason="complete record archived and verified")
    assert discarded.record_count == 0
    assert transport.query("print(smu.source.output)") == "smu.OFF"
    assert float(transport.query("print(smu.source.level)")) == 0.0
    print(asdict(discarded))
finally:
    transport.close()